# MT Model Training - Sequential Fine-Tuning Experiments

This notebook tests whether **sequential fine-tuning** (similar language → baseline) improves MT performance compared to **direct fine-tuning** (baseline only).

## Experimental Design: Sequential Fine-Tuning

For each target language, we create TWO models:

### 1. Baseline Models (Direct Training)
- **Start:** mBART-50 (pretrained)
- **Train:** Distant language → Target language (e.g., `en→tl`)
- **Result:** Baseline Model

### 2. Experimental Models (Sequential Fine-Tuning)
- **Start:** mBART-50 (pretrained)
- **Step 1:** Train on Similar language → Target language (e.g., `bik→tl`)
- **Step 2:** **Continue training** from Step 1 on Distant language → Target language (e.g., `en→tl`)
- **Result:** Experimental Model (with similarity transfer)

### Three Target Languages
1. **Tagalog (tl)**
   - Baseline: `en→tl` only
   - Experimental: `bik→tl` THEN `en→tl`

2. **Ilonggo/Hiligaynon (hil)**
   - Baseline: `en→hil` only
   - Experimental: `msb→hil` THEN `en→hil`

3. **Waray (war)**
   - Baseline: `en→war` only
   - Experimental: `hil→war` THEN `en→war`

## Research Question
**Does "warming up" the model on a similar low-resource language first improve performance on the baseline task?**

Expected: `BLEU(Experimental) > BLEU(Baseline)`

## Install Required Packages

Run this cell first if packages are not installed.

In [ ]:
# Uncomment and run if needed
# !pip install transformers datasets evaluate sacrebleu torch sentencepiece accelerate

## Imports

In [1]:
from transformers import (
    MBartForConditionalGeneration, 
    MBartTokenizerFast, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments
)
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
import torch
from pathlib import Path
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3050


## Configuration

Set up training configurations for all language pairs.

In [8]:
# Model configuration
# NOTE: 'facebook/mbart-base-50' is not a valid Hugging Face model id — use a valid mBART model id below.
# Recommended: 'facebook/mbart-large-50' (common multilingual mBART-50 checkpoint)
MODEL_NAME = "facebook/mbart-large-50"
MAX_LENGTH = 128
BATCH_SIZE = 12 # Optimized for RTX 2050 (4GB VRAM)
LEARNING_RATE = 3e-5
NUM_EPOCHS_STAGE1 = 3  # For similar language training (Stage 1)
NUM_EPOCHS_STAGE2 = 3  # For baseline training (Stage 2)

# Directories
DATA_DIR = Path("../data/splits")
OUTPUT_DIR = Path("../models")
LOGS_DIR = Path("../logs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Experimental configurations
# Each target language has:
# 1. A baseline pair (distant → target)
# 2. A similar pair (similar → target) for Stage 1
EXPERIMENTS = {
    "tagalog": {
        "target": "tl",
        "baseline_pair": "en-tl",  # Distant language
        "similar_pair": "bik-tl",  # Similar language (Stage 1)
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",
        },
        "similar_config": {
            "src_lang": "tl_XX",  # Use tl_XX as proxy for Bikolano
            "tgt_lang": "tl_XX",
        }
    },
    "ilonggo": {
        "target": "hil",
        "baseline_pair": "en-hil",
        "similar_pair": "msb-hil",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",  # Use tl_XX as proxy for Ilonggo
        },
        "similar_config": {
            "src_lang": "tl_XX",  # Use tl_XX as proxy for Masbatenyo
            "tgt_lang": "tl_XX",
        }
    },
    "waray": {
        "target": "war",
        "baseline_pair": "en-war",
        "similar_pair": "hil-war",
        "baseline_config": {
            "src_lang": "en_XX",
            "tgt_lang": "tl_XX",  # Use tl_XX as proxy for Waray
        },
        "similar_config": {
            "src_lang": "tl_XX",  # Use tl_XX as proxy for Ilonggo
            "tgt_lang": "tl_XX",
        }
    }
}

print("Experimental Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs (Stage 1 - Similar): {NUM_EPOCHS_STAGE1}")
print(f"  Epochs (Stage 2 - Baseline): {NUM_EPOCHS_STAGE2}")
print(f"  Max length: {MAX_LENGTH}")
print(f"\nTarget Languages: {len(EXPERIMENTS)}")
for lang, config in EXPERIMENTS.items():
    print(f"  - {lang.capitalize()}: {config['baseline_pair']} (baseline) vs {config['similar_pair']}→{config['baseline_pair']} (sequential)")

Experimental Configuration:
  Model: facebook/mbart-large-50
  Batch size: 12
  Learning rate: 3e-05
  Epochs (Stage 1 - Similar): 3
  Epochs (Stage 2 - Baseline): 3
  Max length: 128

Target Languages: 3
  - Tagalog: en-tl (baseline) vs bik-tl→en-tl (sequential)
  - Ilonggo: en-hil (baseline) vs msb-hil→en-hil (sequential)
  - Waray: en-war (baseline) vs hil-war→en-war (sequential)


## Helper Functions

Functions to load data, preprocess, and evaluate models.

In [3]:
def load_data_for_pair(pair_name):
    """
    Load train and dev splits for a language pair.
    
    Args:
        pair_name: Language pair (e.g., 'en-tl')
    
    Returns:
        DatasetDict with 'train' and 'validation' splits
    """
    src_code, tgt_code = pair_name.split("-")
    pair_dir = DATA_DIR / pair_name
    
    # Load train data
    with open(pair_dir / f"train.{src_code}", "r", encoding="utf-8") as f:
        train_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"train.{tgt_code}", "r", encoding="utf-8") as f:
        train_tgt = [line.strip() for line in f.readlines()]
    
    # Load dev data
    with open(pair_dir / f"dev.{src_code}", "r", encoding="utf-8") as f:
        dev_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{tgt_code}", "r", encoding="utf-8") as f:
        dev_tgt = [line.strip() for line in f.readlines()]
    
    # Create datasets
    train_dataset = Dataset.from_dict({
        "src": train_src,
        "tgt": train_tgt
    })
    
    dev_dataset = Dataset.from_dict({
        "src": dev_src,
        "tgt": dev_tgt
    })
    
    dataset_dict = DatasetDict({
        "train": train_dataset,
        "validation": dev_dataset
    })
    
    print(f"Loaded {pair_name}:")
    print(f"  Train: {len(train_dataset)} examples")
    print(f"  Dev: {len(dev_dataset)} examples")
    
    return dataset_dict


def create_preprocess_function(tokenizer, src_lang, tgt_lang, max_length):
    """
    Create a preprocessing function for tokenization.
    """
    def preprocess(batch):
        # Set source language
        tokenizer.src_lang = src_lang
        
        # Tokenize inputs
        inputs = tokenizer(
            batch["src"], 
            truncation=True, 
            padding="max_length", 
            max_length=max_length
        )
        
        # Set target language
        tokenizer.tgt_lang = tgt_lang
        
        # Tokenize targets
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                batch["tgt"], 
                truncation=True, 
                padding="max_length", 
                max_length=max_length
            )
        
        inputs["labels"] = labels["input_ids"]
        return inputs
    
    return preprocess


def create_compute_metrics(tokenizer):
    """
    Create a function to compute BLEU score during evaluation.
    """
    bleu = evaluate.load("sacrebleu")
    
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        
        # Decode predictions
        if isinstance(preds, tuple):
            preds = preds[0]
        
        # Replace -100 in labels (used for padding)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        
        # Decode
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        # Compute BLEU
        result = bleu.compute(
            predictions=decoded_preds, 
            references=[[label] for label in decoded_labels]
        )
        
        return {"bleu": result["score"]}
    
    return compute_metrics

print("✓ Helper functions defined")

✓ Helper functions defined


## Train a Single Language Pair

Function to train one language pair (can be run for all pairs in a loop).

In [4]:
def train_single_stage(pair_name, config, model_name_or_path, output_subdir, num_epochs, stage_name=""):
    """
    Train an MT model for a single stage (either Stage 1 or baseline).
    
    Args:
        pair_name: Language pair (e.g., 'en-tl', 'bik-tl')
        config: Configuration dictionary with 'src_lang' and 'tgt_lang'
        model_name_or_path: Either MODEL_NAME or path to previously trained model
        output_subdir: Subdirectory name for saving (e.g., 'baseline', 'stage1', 'stage2')
        num_epochs: Number of epochs to train
        stage_name: Description for logging (e.g., "Stage 1: Similar Language")
    
    Returns:
        Training results, metrics, and path to saved model
    """
    print("\n" + "=" * 80)
    print(f"Training: {pair_name.upper()}")
    if stage_name:
        print(f"Stage: {stage_name}")
    print("=" * 80)
    
    # Load tokenizer and model
    print("\n1. Loading model and tokenizer...")
    tokenizer = MBartTokenizerFast.from_pretrained(MODEL_NAME)
    model = MBartForConditionalGeneration.from_pretrained(model_name_or_path)
    print(f"   Loaded from: {model_name_or_path}")
    
    # Load dataset
    print("\n2. Loading dataset...")
    dataset = load_data_for_pair(pair_name)
    
    # Tokenize dataset
    print("\n3. Tokenizing dataset...")
    preprocess_fn = create_preprocess_function(
        tokenizer, 
        config['src_lang'], 
        config['tgt_lang'], 
        MAX_LENGTH
    )
    tokenized_dataset = dataset.map(preprocess_fn, batched=True)
    
    # Set up training arguments
    output_dir = OUTPUT_DIR / output_subdir
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        save_strategy="epoch",
        save_total_limit=2,
        predict_with_generate=True,
        logging_dir=str(LOGS_DIR / output_subdir),
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    
    # Create trainer
    print("\n4. Setting up trainer...")
    compute_metrics_fn = create_compute_metrics(tokenizer)
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_fn,
    )
    
    # Train
    print("\n5. Starting training...")
    train_result = trainer.train()
    
    # Save final model
    print("\n6. Saving model...")
    final_model_path = output_dir / "final_model"
    trainer.save_model(str(final_model_path))
    tokenizer.save_pretrained(str(final_model_path))
    
    # Evaluate on dev set
    print("\n7. Final evaluation on dev set...")
    eval_results = trainer.evaluate()
    
    # Save results
    results = {
        "pair": pair_name,
        "stage": stage_name,
        "model_source": model_name_or_path,
        "config": config,
        "train_results": {
            "train_loss": train_result.training_loss,
            "train_runtime": train_result.metrics["train_runtime"],
            "train_samples_per_second": train_result.metrics["train_samples_per_second"],
        },
        "eval_results": eval_results
    }
    
    results_file = output_dir / "training_results.json"
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ Training complete")
    print(f"  Final BLEU: {eval_results['eval_bleu']:.2f}")
    print(f"  Model saved to: {final_model_path}")
    print(f"  Results saved to: {results_file}")
    
    return results, str(final_model_path)


def train_experiment(target_lang_name, experiment_config):
    """
    Run complete experiment for one target language:
    1. Train baseline model (distant → target)
    2. Train experimental model Stage 1 (similar → target)
    3. Train experimental model Stage 2 (continue with distant → target)
    
    Args:
        target_lang_name: Name of target language (e.g., 'tagalog')
        experiment_config: Configuration dictionary from EXPERIMENTS
    
    Returns:
        Dictionary with all results
    """
    print("\n" + "#" * 80)
    print(f"# EXPERIMENT: {target_lang_name.upper()}")
    print("#" * 80)
    
    baseline_pair = experiment_config['baseline_pair']
    similar_pair = experiment_config['similar_pair']
    
    all_results = {}
    
    # ========================================================================
    # BASELINE MODEL: Train directly on distant → target
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"BASELINE MODEL: {baseline_pair}")
    print(f"Training mBART-50 directly on {baseline_pair} data")
    print(f"{'='*80}")
    
    baseline_results, baseline_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_baseline",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Baseline (Direct Training)"
    )
    all_results['baseline'] = baseline_results
    
    # ========================================================================
    # EXPERIMENTAL MODEL - STAGE 1: Train on similar → target
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL MODEL - STAGE 1: {similar_pair}")
    print(f"Training mBART-50 on similar language data ({similar_pair})")
    print(f"{'='*80}")
    
    stage1_results, stage1_model_path = train_single_stage(
        pair_name=similar_pair,
        config=experiment_config['similar_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_experimental_stage1",
        num_epochs=NUM_EPOCHS_STAGE1,
        stage_name="Stage 1: Similar Language"
    )
    all_results['experimental_stage1'] = stage1_results
    
    # ========================================================================
    # EXPERIMENTAL MODEL - STAGE 2: Continue training on distant → target
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL MODEL - STAGE 2: {baseline_pair}")
    print(f"Continuing from Stage 1 model, now training on {baseline_pair} data")
    print(f"{'='*80}")
    
    stage2_results, stage2_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=stage1_model_path,  # Load from Stage 1!
        output_subdir=f"{target_lang_name}_experimental_stage2",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Stage 2: Baseline (After Similar)"
    )
    all_results['experimental_stage2'] = stage2_results
    
    # ========================================================================
    # SUMMARY
    # ========================================================================
    print("\n" + "=" * 80)
    print(f"EXPERIMENT COMPLETE: {target_lang_name.upper()}")
    print("=" * 80)
    print(f"\nBaseline Model ({baseline_pair} only):")
    print(f"  BLEU: {all_results['baseline']['eval_results']['eval_bleu']:.2f}")
    print(f"\nExperimental Model ({similar_pair} → {baseline_pair}):")
    print(f"  Stage 1 ({similar_pair}): {all_results['experimental_stage1']['eval_results']['eval_bleu']:.2f}")
    print(f"  Stage 2 ({baseline_pair}): {all_results['experimental_stage2']['eval_results']['eval_bleu']:.2f}")
    
    improvement = all_results['experimental_stage2']['eval_results']['eval_bleu'] - all_results['baseline']['eval_results']['eval_bleu']
    print(f"\nImprovement: {improvement:+.2f} BLEU points")
    
    if improvement > 0:
        print("✓ Sequential fine-tuning IMPROVED performance")
    elif improvement < 0:
        print("✗ Sequential fine-tuning DEGRADED performance")
    else:
        print("= No difference in performance")
    
    # Save experiment summary
    summary_file = OUTPUT_DIR / f"{target_lang_name}_experiment_summary.json"
    with open(summary_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
    
    return all_results

print("✓ Training functions defined")

✓ Training functions defined


## Run Single Experiment

Test with one target language first to verify the setup.

In [9]:
#Run one experiment (uncomment to test)
target_lang = "tagalog"
results = train_experiment(target_lang, EXPERIMENTS[target_lang])


################################################################################
# EXPERIMENT: TAGALOG
################################################################################

BASELINE MODEL: en-tl
Training mBART-50 directly on en-tl data

Training: EN-TL
Stage: Baseline (Direct Training)

1. Loading model and tokenizer...



################################################################################
# EXPERIMENT: TAGALOG
################################################################################

BASELINE MODEL: en-tl
Training mBART-50 directly on en-tl data

Training: EN-TL
Stage: Baseline (Direct Training)

1. Loading model and tokenizer...


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]


################################################################################
# EXPERIMENT: TAGALOG
################################################################################

BASELINE MODEL: en-tl
Training mBART-50 directly on en-tl data

Training: EN-TL
Stage: Baseline (Direct Training)

1. Loading model and tokenizer...


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

...

Map:   0%|          | 0/2358 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/294 [00:00<?, ? examples/s]

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## Run All Experiments

Run all three experiments. **WARNING: This will take several hours!**

For each target language, this will:
1. Train baseline model (distant → target)
2. Train Stage 1 (similar → target)
3. Train Stage 2 (continue with distant → target)

Total: 9 training runs (3 per target language × 3 target languages)

In [ ]:
# Run all experiments (uncomment to run)
# all_experiment_results = {}
# 
# for target_lang, exp_config in EXPERIMENTS.items():
#     try:
#         results = train_experiment(target_lang, exp_config)
#         all_experiment_results[target_lang] = results
#     except Exception as e:
#         print(f"\n❌ Error in {target_lang} experiment: {e}")
#         import traceback
#         traceback.print_exc()
#         continue
# 
# # Save overall summary
# final_summary_file = OUTPUT_DIR / "all_experiments_summary.json"
# with open(final_summary_file, "w") as f:
#     json.dump(all_experiment_results, f, indent=2)
# 
# print("\n" + "#" * 80)
# print("# ALL EXPERIMENTS COMPLETE")
# print("#" * 80)
# print(f"\nFinal summary saved to: {final_summary_file}")
# 
# # Print comparison table
# print("\n" + "=" * 80)
# print("RESULTS SUMMARY")
# print("=" * 80)
# print(f"{'Target Language':<20} {'Baseline BLEU':<15} {'Sequential BLEU':<15} {'Improvement':<15}")
# print("-" * 80)
# for target_lang, results in all_experiment_results.items():
#     baseline_bleu = results['baseline']['eval_results']['eval_bleu']
#     sequential_bleu = results['experimental_stage2']['eval_results']['eval_bleu']
#     improvement = sequential_bleu - baseline_bleu
#     print(f"{target_lang.capitalize():<20} {baseline_bleu:<15.2f} {sequential_bleu:<15.2f} {improvement:+.2f}")

## Summary and Next Steps

### What This Notebook Does

This notebook implements **sequential fine-tuning experiments** to test whether "warming up" a model on similar language data improves performance on the baseline task.

#### For Each Target Language (Tagalog, Ilonggo, Waray):

**Baseline Model:**
1. Load mBART-50
2. Train on distant→target (e.g., `en→tl`)
3. Evaluate

**Experimental Model (Sequential):**
1. Load mBART-50
2. **Stage 1:** Train on similar→target (e.g., `bik→tl`)
3. **Stage 2:** Load Stage 1 model, continue training on distant→target (e.g., `en→tl`)
4. Evaluate

### Output Structure
```
models/
  ├── tagalog_baseline/
  │   └── final_model/          # Baseline: en→tl only
  ├── tagalog_experimental_stage1/
  │   └── final_model/          # Stage 1: bik→tl
  ├── tagalog_experimental_stage2/
  │   └── final_model/          # Stage 2: bik→tl THEN en→tl (FINAL)
  ├── tagalog_experiment_summary.json
  │
  ├── ilonggo_baseline/
  ├── ilonggo_experimental_stage1/
  ├── ilonggo_experimental_stage2/
  ├── ilonggo_experiment_summary.json
  │
  ├── waray_baseline/
  ├── waray_experimental_stage1/
  ├── waray_experimental_stage2/
  ├── waray_experiment_summary.json
  │
  └── all_experiments_summary.json
```

### Key Difference from Previous Design

**OLD (Incorrect):** Six independent models trained separately
- `en→tl`, `bik→tl`, `en→hil`, `msb→hil`, `en→war`, `hil→war`
- No sequential training

**NEW (Correct):** Three experiments with baseline + sequential fine-tuning
- Tagalog: `en→tl` vs. (`bik→tl` → `en→tl`)
- Ilonggo: `en→hil` vs. (`msb→hil` → `en→hil`)
- Waray: `en→war` vs. (`hil→war` → `en→war`)

### Research Question

**Does sequential fine-tuning from a similar language improve MT performance?**

Expected Result: `BLEU(Sequential) > BLEU(Baseline)`

### Training Notes

- **Total Training Runs:** 9 (3 per target language)
- **GPU Highly Recommended:** RTX 2050 (4GB) should work with `BATCH_SIZE=2`
- **Time Estimate:** ~1 hour per training run on GPU = ~9 hours total
- **Memory:** Monitor VRAM usage; reduce batch size to 1 if OOM errors occur

### After Training: Evaluation

1. **Load Both Models** (baseline and experimental_stage2)
2. **Evaluate on Test Set** (not dev set)
3. **Compute BLEU Scores**
4. **Statistical Significance Testing**
   - Paired bootstrap resampling
   - Test if improvement is statistically significant

### Expected Analysis

For each target language:
- Compare `BLEU(Baseline)` vs. `BLEU(Experimental)`
- Calculate improvement: `Δ BLEU = Experimental - Baseline`
- Perform significance test
- Analyze: Does similarity transfer help? By how much?

### Tips for Running

1. **Start with ONE experiment** (e.g., Tagalog) to verify setup
2. **Monitor GPU memory** - reduce batch size if needed
3. **Check intermediate results** - look at Stage 1 BLEU to see if similar language training is working
4. **Save checkpoints** - models are saved after each epoch
5. **Be patient** - sequential training takes time but tests a much stronger hypothesis